# 03 — Train Models

Trains (or resumes) **YOLOv8 multi-class**, **YOLOv8 single-class**, **ResNet50**, and the **convolutional autoencoder**.

All checkpoints persist in Drive. Re-running skips already-trained models.

> Requires NB02 to have been run first.

In [ ]:
import os
FLAG = "/content/.deps_ok_karyotype"
if not os.path.exists(FLAG):
    !pip -q install --no-cache-dir --upgrade --force-reinstall \
        "numpy==2.0.2" "pandas==2.2.2" "opencv-python==4.10.0.84" \
        "ultralytics==8.*" "torch" "torchvision" "tqdm==4.*" "matplotlib==3.*"
    open(FLAG,"w").write("ok")
    raise SystemExit("✅ Restart runtime, then re-run.")
print("✅ Dependencies OK")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil, torch
from pathlib import Path

ROOT    = "/content/drive/MyDrive/54816"
OUT_DIR = f"{ROOT}/e2e_outputs_run"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Hyperparameters EXACTLY as published (Scientific Reports) ─────────
# YOLO models (all three: overlap, single-class, 24-class)
YOLO_EPOCHS  = 40        # paper Section 2.3.2 / 2.4 / 3.1: 40 epochs
YOLO_IMGSZ   = 640
YOLO_BATCH   = 32        # paper: batch size 32
YOLO_LR      = 0.001     # paper: learning_rate = 0.001

# ResNet-50 (paper Section 3.2)
RESNET_EPOCHS = 100      # up to 100 epochs (EarlyStopping controls actual stop)
RESNET_BATCH  = 32       # paper batch size 32
RESNET_LR     = 0.001    # Adam lr=0.001

# Dataset B — Q-band isolated chromosomes (Poletti et al. 2012)
# Place the dataset in Drive at this path (24 class sub-folders inside)
DATASET_B_DIR = "/content/drive/MyDrive/dataset_b_qband"  # <-- update if needed

YOLO_DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", YOLO_DEVICE, "| CUDA:", torch.cuda.is_available())
print("YOLO  : epochs=", YOLO_EPOCHS, "batch=", YOLO_BATCH, "lr=", YOLO_LR)
print("ResNet: epochs=", RESNET_EPOCHS, "batch=", RESNET_BATCH, "lr=", RESNET_LR)


In [ ]:
# ── Train YOLOv8 multi-class (24 chromosome pairs) ──────────────────
# Paper Section 3.1: lr=0.001, batch=32, epochs=40
from ultralytics import YOLO

yaml_multi  = os.path.join(OUT_DIR,"yolo_multiclass","data.yaml")
alias_multi = os.path.join(OUT_DIR,"yolo_multiclass_best.pt")
run_name_m  = "train_yolo_multiclass"

if not os.path.isfile(yaml_multi):
    raise FileNotFoundError(f"Run NB02 first: {yaml_multi}")

if os.path.isfile(alias_multi):
    print("[OK] Multi-class weights already exist:", alias_multi)
else:
    run_dir = os.path.join(OUT_DIR, run_name_m)
    last_pt = os.path.join(run_dir,"weights","last.pt")
    best_pt = os.path.join(run_dir,"weights","best.pt")
    model   = YOLO(last_pt if os.path.isfile(last_pt) else "yolov8n.pt")
    model.train(
        data=yaml_multi, epochs=YOLO_EPOCHS, imgsz=YOLO_IMGSZ,
        batch=YOLO_BATCH, lr0=YOLO_LR,
        device=YOLO_DEVICE, project=OUT_DIR, name=run_name_m,
        exist_ok=True, resume=os.path.isfile(last_pt)
    )
    shutil.copy(best_pt, alias_multi)
    print("[DONE] Multi-class model:", alias_multi)


In [ ]:
# ── Train YOLOv8 single-class (chromosome detector for Pipeline B) ──
# Paper Section 2.4: same hyperparameters as overlap model
yaml_one   = os.path.join(OUT_DIR,"yolo_oneclass","data.yaml")
alias_one  = os.path.join(OUT_DIR,"yolo_oneclass_best.pt")
run_name_o = "train_yolo_oneclass"

if os.path.isfile(alias_one):
    print("[OK] Single-class weights already exist:", alias_one)
else:
    run_dir = os.path.join(OUT_DIR, run_name_o)
    last_pt = os.path.join(run_dir,"weights","last.pt")
    best_pt = os.path.join(run_dir,"weights","best.pt")
    model   = YOLO(last_pt if os.path.isfile(last_pt) else "yolov8n.pt")
    model.train(
        data=yaml_one, epochs=YOLO_EPOCHS, imgsz=YOLO_IMGSZ,
        batch=YOLO_BATCH, lr0=YOLO_LR,
        device=YOLO_DEVICE, project=OUT_DIR, name=run_name_o,
        exist_ok=True, resume=os.path.isfile(last_pt)
    )
    shutil.copy(best_pt, alias_one)
    print("[DONE] Single-class model:", alias_one)


In [ ]:
# ── Train ResNet50 pair classifier ──────────────────────────────────
# Paper Section 3.2: TF/Keras, Dataset B (Q-band), 75/25 split via
# ImageDataGenerator, Adam lr=0.001, EarlyStopping patience=15,
# ReduceLROnPlateau factor=0.02 patience=3, freeze first ~50 layers,
# up to 100 epochs.
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

RESNET_SAVE = f"{ROOT}/weights_resnet/ResNet50_cromosomas_model.keras"
os.makedirs(os.path.dirname(RESNET_SAVE), exist_ok=True)

if os.path.isfile(RESNET_SAVE):
    print("[OK] ResNet already exists:", RESNET_SAVE)
else:
    if not os.path.isdir(DATASET_B_DIR):
        raise FileNotFoundError(
            f"Dataset B not found at {DATASET_B_DIR}.\n"
            "Download the Q-band corpus (Poletti et al. 2012) and place it there."
        )

    # ── Data generators with augmentation (paper Section 3.2) ──────
    datagen = ImageDataGenerator(
        preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
        rotation_range=30,        # paper: rotations ≤ 30°
        zoom_range=0.20,           # paper: zoom ≤ 20%
        horizontal_flip=True,      # paper: horizontal flips
        width_shift_range=0.10,    # paper: shifts ≤ 10%
        height_shift_range=0.10,
        validation_split=0.25,     # paper: 75% / 25%
    )

    train_gen = datagen.flow_from_directory(
        DATASET_B_DIR, target_size=(224,224), batch_size=RESNET_BATCH,
        class_mode="categorical", subset="training"
    )
    val_gen = datagen.flow_from_directory(
        DATASET_B_DIR, target_size=(224,224), batch_size=RESNET_BATCH,
        class_mode="categorical", subset="validation"
    )

    # Save class index mapping for inference
    json.dump(train_gen.class_indices,
              open(os.path.join(OUT_DIR,"resnet_class_indices.json"),"w"), indent=2)
    print("[INFO] Classes:", len(train_gen.class_indices), "→", list(train_gen.class_indices.keys())[:6])

    # ── Model architecture (paper Section 3.2) ──────────────────────
    base = ResNet50(weights="imagenet", include_top=True)
    x    = base.layers[-2].output                           # features before final dense
    out  = Dense(len(train_gen.class_indices), activation="softmax")(x)
    model = Model(inputs=base.input, outputs=out)

    # Freeze first ~50 layers
    for layer in base.layers[:50]:
        layer.trainable = False

    model.compile(
        optimizer=Adam(learning_rate=RESNET_LR),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    # ── Callbacks ──────────────────────────────────────────────────
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
        ModelCheckpoint(RESNET_SAVE, monitor="val_loss", save_best_only=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.02, patience=3,
                          min_lr=1e-6, verbose=1),
    ]

    history = model.fit(
        train_gen, validation_data=val_gen,
        epochs=RESNET_EPOCHS, callbacks=callbacks
    )

    # Plot training curves
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1,2,figsize=(12,4))
    axes[0].plot(history.history["accuracy"],    label="Train Acc")
    axes[0].plot(history.history["val_accuracy"],label="Val Acc")
    axes[0].set_title("Accuracy"); axes[0].legend()
    axes[1].plot(history.history["loss"],    label="Train Loss")
    axes[1].plot(history.history["val_loss"],label="Val Loss")
    axes[1].set_title("Loss"); axes[1].legend()
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR,"resnet_curves.png")); plt.show()
    print("[DONE] ResNet50 saved:", RESNET_SAVE)


In [ ]:
# ── Train Convolutional Autoencoder (structural anomaly detection) ──
# Paper Section 6:
#   Input:     28×28×1  grayscale
#   Encoder:   Conv2D(128) → MaxPool → Conv2D(64) → MaxPool → Conv2D(32) → MaxPool  → 4×4×32
#   Decoder:   Conv2D(32)  → UpSamp  → Conv2D(64) → UpSamp  → Conv2D(128) → UpSamp → Conv2D(1, sigmoid)
#   Loss:      MSE
#   Threshold: τ = 0.0148 (paper Table 2)
#   Training:  NORMAL images ONLY from Dataset A (exclude numerical + structural anomaly cases)

import tensorflow as tf
from tensorflow.keras import layers, Model
import glob, cv2
import numpy as np

AE_SAVE_PATH = f"{ROOT}/weights_autoencoder/autoencoder_model.keras"
AE_CROPS_DIR = os.path.join(OUT_DIR, "ae_crops_normal")
AE_IMG_SIZE  = (28, 28)       # paper: 28×28×1 grayscale
AE_EPOCHS    = 50
AE_BATCH     = 32
CSV_NORMAL   = f"{ROOT}/normal.csv"
YOLO24_BEST  = os.path.join(OUT_DIR,"yolo_multiclass_best.pt")

os.makedirs(os.path.dirname(AE_SAVE_PATH), exist_ok=True)
os.makedirs(AE_CROPS_DIR, exist_ok=True)

if os.path.isfile(AE_SAVE_PATH):
    print("[OK] Autoencoder already exists:", AE_SAVE_PATH)
else:
    # ── Step A: Extract grayscale crops from NORMAL cases ───────────
    import torch, pandas as pd
    from ultralytics import YOLO
    from tqdm import tqdm

    if not os.path.isfile(YOLO24_BEST):
        raise FileNotFoundError(f"Train YOLO first: {YOLO24_BEST}")
    if not os.path.isfile(CSV_NORMAL):
        raise FileNotFoundError(f"normal.csv not found: {CSV_NORMAL}")

    device   = 0 if torch.cuda.is_available() else "cpu"
    yolo     = YOLO(YOLO24_BEST)
    IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".tif",".tiff"}

    normal_df    = pd.read_csv(CSV_NORMAL, header=None)
    normal_stems = set(Path(str(v)).stem for v in normal_df.iloc[:,0])
    print(f"[INFO] Normal cases: {len(normal_stems)}")

    MULTI_IMG_DIR = f"{ROOT}/24_chromosomes_object/JEPG"
    all_imgs = []
    for ext in IMG_EXTS: all_imgs += list(Path(MULTI_IMG_DIR).rglob(f"*{ext}"))
    normal_imgs = [p for p in all_imgs if p.stem in normal_stems]
    print(f"[INFO] Normal images found: {len(normal_imgs)}")

    crop_count = 0
    for img_path in tqdm(normal_imgs, desc="Extracting AE crops"):
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None: continue
        r = yolo(str(img_path), verbose=False, device=device, conf=0.25)[0]
        if r.boxes is None or len(r.boxes)==0: continue
        for bi, box in enumerate(r.boxes):
            x1,y1,x2,y2 = [int(v) for v in box.xyxy[0].tolist()]
            H,W = img_bgr.shape[:2]
            x1=max(0,x1); y1=max(0,y1); x2=min(W,x2); y2=min(H,y2)
            if x2<=x1 or y2<=y1: continue
            crop_bgr = img_bgr[y1:y2,x1:x2]
            # Convert to GRAYSCALE (paper uses 28×28×1)
            crop_gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
            out_f = os.path.join(AE_CROPS_DIR, f"{img_path.stem}_{bi:03d}.png")
            cv2.imwrite(out_f, crop_gray)
            crop_count += 1
    print(f"[DONE] Extracted {crop_count} grayscale crops")

    # ── Step B: Load and normalise ──────────────────────────────────
    imgs = []
    for f in glob.glob(os.path.join(AE_CROPS_DIR,"*.png")):
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        img = cv2.resize(img, AE_IMG_SIZE).astype(np.float32)/255.0
        imgs.append(img[..., np.newaxis])   # (28,28,1)
    X = np.stack(imgs)
    print(f"[INFO] AE dataset: {X.shape}")

    # ── Step C: Build autoencoder (paper Figure 4) ──────────────────
    inp = layers.Input(shape=(28,28,1))
    # Encoder
    x = layers.Conv2D(128,3,activation='relu',padding='same')(inp)
    x = layers.MaxPooling2D(2,padding='same')(x)         # 14×14×128
    x = layers.Conv2D(64, 3,activation='relu',padding='same')(x)
    x = layers.MaxPooling2D(2,padding='same')(x)         # 7×7×64
    x = layers.Conv2D(32, 3,activation='relu',padding='same')(x)
    encoded = layers.MaxPooling2D(2,padding='same')(x)   # 4×4×32
    # Decoder
    x = layers.Conv2D(32, 3,activation='relu',padding='same')(encoded)
    x = layers.UpSampling2D(2)(x)                        # 8×8×32
    x = layers.Conv2D(64, 3,activation='relu',padding='same')(x)
    x = layers.UpSampling2D(2)(x)                        # 16×16×64
    x = layers.Conv2D(128,3,activation='relu',padding='same')(x)
    x = layers.UpSampling2D(2)(x)                        # 32×32×128
    decoded = layers.Conv2D(1,3,activation='sigmoid',padding='same')(x)
    decoded = layers.Cropping2D(((2,2),(2,2)))(decoded)  # 32→28

    ae = Model(inp, decoded, name="autoencoder")
    ae.compile(optimizer='adam', loss='mse')
    ae.summary()

    # ── Step D: Train ───────────────────────────────────────────────
    ae.fit(X, X, epochs=AE_EPOCHS, batch_size=AE_BATCH,
           validation_split=0.10,
           callbacks=[tf.keras.callbacks.EarlyStopping(
               patience=5, restore_best_weights=True)])
    ae.save(AE_SAVE_PATH)
    print("[DONE] Autoencoder saved:", AE_SAVE_PATH)
    print("Published threshold τ = 0.0148 (paper Table 2, Section 8.5)")
    print("Update configs/config.json ae_threshold if retraining on different data.")
